<a href="https://colab.research.google.com/github/PALAK7890/ab_testing/blob/main/marketing_ab_testing_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")

print("Path to dataset files:", path)

100%|██████████| 5.23M/5.23M [00:00<00:00, 5.56MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/faviovaz/marketing-ab-testing/versions/1


In [2]:
import pandas as pd
import numpy as np
from math import sqrt, asin
from scipy.stats import norm
import os

path = "/root/.cache/kagglehub/datasets/faviovaz/marketing-ab-testing/versions/1"
print(os.listdir(path))

['marketing_AB.csv']


In [3]:
file_path = path + "/marketing_AB.csv"
df=pd.read_csv(file_path)

In [4]:
df.head()

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


In [5]:
df.isnull().sum()

,0
Unnamed: 0,0
user id,0
test group,0
converted,0
total ads,0
most ads day,0
most ads hour,0


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [8]:
# Group distribution
print("\nTest Group Distribution:")
print(df["test_group"].value_counts())

# Conversion distribution
print("\nConversion Distribution:")
print(df["converted"].value_counts())

# Conversion rate by group
conversion_summary = df.groupby("test_group")["converted"].mean()

print("\nConversion Rate by Group:")
print(conversion_summary)


Test Group Distribution:
test_group
ad     564577
psa     23524
Name: count, dtype: int64

Conversion Distribution:
converted
False    573258
True      14843
Name: count, dtype: int64

Conversion Rate by Group:
test_group
ad     0.025547
psa    0.017854
Name: converted, dtype: float64


In [9]:
control = df[df["test_group"] == "psa"]
treatment = df[df["test_group"] == "ad"]

n1 = len(control)
x1 = control["converted"].sum()
p1 = x1 / n1

n2 = len(treatment)
x2 = treatment["converted"].sum()
p2 = x2 / n2
p_pool = (x1 + x2) / (n1 + n2)

se = sqrt(p_pool * (1 - p_pool) * ((1/n1) + (1/n2)))

z = (p2 - p1) / se

p_value = 2 * (1 - norm.cdf(abs(z)))

print("Z-score:", z)
print("P-value:", p_value)

Z-score: 7.3700781265454145
P-value: 1.7053025658242404e-13


In [10]:
se_diff = sqrt((p1 * (1 - p1) / n1) + (p2 * (1 - p2) / n2))

z_critical = norm.ppf(0.975)

diff = p2 - p1

ci_low = diff - z_critical * se_diff
ci_high = diff + z_critical * se_diff

print("95% CI:", ci_low, "to", ci_high)

95% CI: 0.0059509324316110055 to 0.009433973952792028


In [11]:

cohens_h = 2 * (asin(sqrt(p2)) - asin(sqrt(p1)))

print("Cohen's h:", cohens_h)

Cohen's h: 0.053002578606030915


In [12]:
np.random.seed(42)

# create ad_type only for ad group
ad_types = ["Static", "Video", "Carousel"]

df["ad_type"] = np.where(
    df["test_group"] == "ad",
    np.random.choice(ad_types, size=len(df), p=[0.4, 0.35, 0.25]),
    "PSA"
)

# assign a realistic cost per ad exposure
cost_map = {
    "Static": 1.5,
    "Video": 2.8,
    "Carousel": 2.2,
    "PSA": 0.8
}

df["cost_per_ad"] = df["ad_type"].map(cost_map)

# create clean copy of total_ads if needed
df["total_ads"] = df["total_ads"].astype(int)

# estimate total spend per user
df["total_cost"] = df["total_ads"] * df["cost_per_ad"]

# simulate revenue only for users who converted
def simulate_revenue(row):
    if row["converted"] == False:
        return 0
    if row["ad_type"] == "Video":
        return np.random.randint(600, 1200)
    elif row["ad_type"] == "Carousel":
        return np.random.randint(450, 900)
    elif row["ad_type"] == "Static":
        return np.random.randint(300, 700)
    else:
        return np.random.randint(200, 500)

df["revenue"] = df.apply(simulate_revenue, axis=1)

# calculate profit and ROI
df["profit"] = df["revenue"] - df["total_cost"]
df["roi"] = np.where(df["total_cost"] > 0, df["profit"] / df["total_cost"], 0)

print(df[["test_group", "ad_type", "converted", "total_ads", "total_cost", "revenue", "profit", "roi"]].head())

  test_group   ad_type  converted  total_ads  total_cost  revenue  profit  roi
0         ad    Static      False        130       195.0        0  -195.0 -1.0
1         ad  Carousel      False         93       204.6        0  -204.6 -1.0
2         ad     Video      False         21        58.8        0   -58.8 -1.0
3         ad     Video      False        355       994.0        0  -994.0 -1.0
4         ad    Static      False        276       414.0        0  -414.0 -1.0


In [13]:
ad_perf = df[df["ad_type"] != "PSA"].groupby("ad_type").agg(
    users=("user_id", "count"),
    conversions=("converted", "sum"),
    total_cost=("total_cost", "sum"),
    total_revenue=("revenue", "sum"),
    total_profit=("profit", "sum")
).reset_index()

ad_perf["conversion_rate"] = ad_perf["conversions"] / ad_perf["users"]
ad_perf["roi"] = ad_perf["total_profit"] / ad_perf["total_cost"]

print(ad_perf.sort_values("roi", ascending=False))

    ad_type   users  conversions  total_cost  total_revenue  total_profit  \
1    Static  225730         5783   8379936.0        2864901    -5515035.0   
2     Video  197771         5001  13788210.8        4527121    -9261089.8   
0  Carousel  141076         3639   7708175.2        2462154    -5246021.2   

   conversion_rate       roi  
1         0.025619 -0.658124  
2         0.025287 -0.671667  
0         0.025795 -0.680579  


In [14]:
monthly_budget = 50000

budget_table = ad_perf.copy()

# shift ROI to positive weights for allocation
min_roi = budget_table["roi"].min()

budget_table["roi_weight"] = budget_table["roi"] - min_roi + 0.01

budget_table["allocation_pct"] = budget_table["roi_weight"] / budget_table["roi_weight"].sum()

budget_table["recommended_budget"] = budget_table["allocation_pct"] * monthly_budget

budget_table["equal_budget"] = monthly_budget / len(budget_table)

print(budget_table[["ad_type", "roi", "equal_budget", "recommended_budget"]].sort_values("recommended_budget", ascending=False))

    ad_type       roi  equal_budget  recommended_budget
1    Static -0.658124  16666.666667        26443.575887
2     Video -0.671667  16666.666667        15408.686047
0  Carousel -0.680579  16666.666667         8147.738066


In [15]:
dashboard_df = df.copy()

# Clean column names
dashboard_df.columns = dashboard_df.columns.str.lower().str.replace(" ", "_")

# Convert converted to int
dashboard_df["converted"] = dashboard_df["converted"].astype(int)

# Add readable group names
dashboard_df["group_label"] = dashboard_df["test_group"].map({
    "ad": "Ad Campaign",
    "psa": "PSA"
})

# Create time period from hour
def get_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

dashboard_df["time_period"] = dashboard_df["most_ads_hour"].apply(get_time_period)

# Create ad exposure buckets
dashboard_df["ad_exposure_bucket"] = pd.cut(
    dashboard_df["total_ads"],
    bins=[0, 5, 10, 20, 50, 100, dashboard_df["total_ads"].max()],
    labels=["1-5", "6-10", "11-20", "21-50", "51-100", "100+"],
    include_lowest=True
)

# Final columns (clean + useful only)
dashboard_df = dashboard_df[[
    "user_id",
    "test_group",
    "group_label",
    "converted",
    "total_ads",
    "ad_exposure_bucket",
    "most_ads_day",
    "most_ads_hour",
    "time_period"
]]

# Save single dataset
dashboard_df.to_csv("ab_test_dashboard_ready.csv", index=False)

print("Final dataset ready: ab_test_dashboard_ready.csv")

Final dataset ready: ab_test_dashboard_ready.csv


In [16]:
# from google.colab import files
# files.download("ab_test_dashboard_ready.csv")

In [17]:
import pandas as pd
import numpy as np

# FINAL MASTER DATASET


final_df = df.copy()

# Clean column names
final_df.columns = final_df.columns.str.lower().str.replace(" ", "_")

# Convert converted to int
final_df["converted"] = final_df["converted"].astype(int)

# DASHBOARD 1 FEATURES

# Group labels
final_df["group_label"] = final_df["test_group"].map({
    "ad": "Ad Campaign",
    "psa": "PSA"
})

# Time period
def get_time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

final_df["time_period"] = final_df["most_ads_hour"].apply(get_time_period)

# Exposure buckets
final_df["ad_exposure_bucket"] = pd.cut(
    final_df["total_ads"],
    bins=[0, 5, 10, 20, 50, 100, final_df["total_ads"].max()],
    labels=["1-5", "6-10", "11-20", "21-50", "51-100", "100+"],
    include_lowest=True
)

# DASHBOARD 2 FEATURES (ROI)

np.random.seed(42)

# Ad types
ad_types = ["Static", "Video", "Carousel"]

final_df["ad_type"] = np.where(
    final_df["test_group"] == "ad",
    np.random.choice(ad_types, size=len(final_df), p=[0.4, 0.35, 0.25]),
    "PSA"
)

# Cost per ad
cost_map = {
    "Static": 1.5,
    "Video": 2.8,
    "Carousel": 2.2,
    "PSA": 0.8
}

final_df["cost_per_ad"] = final_df["ad_type"].map(cost_map)

# Total cost
final_df["total_cost"] = final_df["total_ads"] * final_df["cost_per_ad"]

# Revenue simulation
def revenue_func(row):
    if row["converted"] == 0:
        return 0
    if row["ad_type"] == "Video":
        return np.random.randint(600, 1200)
    elif row["ad_type"] == "Carousel":
        return np.random.randint(450, 900)
    elif row["ad_type"] == "Static":
        return np.random.randint(300, 700)
    else:
        return np.random.randint(200, 500)

final_df["revenue"] = final_df.apply(revenue_func, axis=1)

# Profit & ROI
final_df["profit"] = final_df["revenue"] - final_df["total_cost"]
final_df["roi"] = final_df["profit"] / final_df["total_cost"]

# DASHBOARD 3 SUPPORT

# Revenue per conversion (useful for analysis)
final_df["rev_per_conversion"] = np.where(
    final_df["converted"] == 1,
    final_df["revenue"],
    0
)

# FINAL COLUMN ORDER

final_df = final_df[[
    "user_id",
    "test_group",
    "group_label",
    "converted",
    "total_ads",
    "ad_exposure_bucket",
    "most_ads_day",
    "most_ads_hour",
    "time_period",
    "ad_type",
    "cost_per_ad",
    "total_cost",
    "revenue",
    "profit",
    "roi",
    "rev_per_conversion"
]]

final_df.to_csv("ab_test_master_dataset.csv", index=False)

print(" MASTER DATASET READY: ab_test_master_dataset.csv")

✅ MASTER DATASET READY: ab_test_master_dataset.csv


In [18]:
from google.colab import files
files.download("ab_test_master_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>